# **Insurance Claims Data Cleaning**

## 1. Importing Libraries and Loading Data

In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

import sys
sys.path.append(str(Path.cwd().parent))

from scripts.paths import RAW_DIR, CLEANED_DIR

df = pd.read_csv(RAW_DIR / "11_insurance_claims_dirty.csv")

## 2. Initial Data Inspection

In [5]:
df.head()

,Claim_ID,Patient_ID,Encounter_ID,Claim_Date,Payer,Claimed_Amount,Approved_Amount,Rejected_Amount,Claim_Status
0,C800000,P106188,E203019,2026-02-13,Corporate,156097.06,148936.23,7160.83,Rejected
1,C800001,P106653,E214050,2026-04-28,Corporate,18925.52,18407.29,518.23,NaN
2,C800002,P106159,E214940,2026-06-05,Self Pay,243501.28,110166.21,133335.07,Partially Approved
3,C800003,P106280,E210273,2026-07-06,ICICI Lombard,171669.61,107189.01,64480.60,Pending
4,C800004,P100535,E214144,2026-02-11,Ayushman Bharat,88288.84,80392.10,7896.74,Approved


In [6]:
print("Shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isna().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nBasic statistics:")
display(df.describe(include="all"))

Shape: (9000, 9)

Column names:
['Claim_ID', 'Patient_ID', 'Encounter_ID', 'Claim_Date', 'Payer', 'Claimed_Amount', 'Approved_Amount', 'Rejected_Amount', 'Claim_Status']

Data types:
Claim_ID            object
Patient_ID          object
Encounter_ID        object
Claim_Date          object
Payer               object
Claimed_Amount     float64
Approved_Amount    float64
Rejected_Amount    float64
Claim_Status        object
dtype: object

Missing values:
Claim_ID              0
Patient_ID            0
Encounter_ID          0
Claim_Date            0
Payer                 0
Claimed_Amount        0
Approved_Amount       0
Rejected_Amount       0
Claim_Status       1568
dtype: int64

Duplicate rows: 0

Basic statistics:


,Claim_ID,Patient_ID,Encounter_ID,Claim_Date,Payer,Claimed_Amount,Approved_Amount,Rejected_Amount,Claim_Status
count,9000,9000,9000,9000,9000,9000.000000,9000.000000,9000.000000,7432
unique,9000,5353,6756,601,7,NaN,NaN,NaN,5
top,C800000,P104399,E205238,2026-06-21,Ayushman Bharat,NaN,NaN,NaN,Approved
freq,1,8,5,25,1322,NaN,NaN,NaN,1538
mean,NaN,NaN,NaN,NaN,NaN,151399.915138,109677.666168,41722.248970,NaN
std,NaN,NaN,NaN,NaN,NaN,86236.274853,68393.075413,36480.311286,NaN
min,NaN,NaN,NaN,NaN,NaN,505.730000,299.810000,3.320000,NaN
25%,NaN,NaN,NaN,NaN,NaN,77654.327500,52454.912500,11541.700000,NaN
50%,NaN,NaN,NaN,NaN,NaN,151440.380000,104136.590000,31780.195000,NaN
75%,NaN,NaN,NaN,NaN,NaN,225477.070000,158335.897500,62992.417500,NaN


## 3. Audit of Specific Columns

In [7]:
# Claim_ID
print("Duplicate Claim_ID:", df["Claim_ID"].duplicated().sum())
print(
    "Invalid Claim_ID format:",
    (
        df["Claim_ID"].astype("string")
        .notna()
        & ~df["Claim_ID"].astype("string").str.fullmatch(r"C\d{6}", na=False)
    ).sum()
)

# Patient_ID
print(
    "Invalid Patient_ID format:",
    (
        df["Patient_ID"].astype("string").notna()
        & ~df["Patient_ID"].astype("string").str.fullmatch(r"P\d{6}", na=False)
    ).sum()
)

# Encounter_ID
print(
    "Invalid Encounter_ID format:",
    (
        df["Encounter_ID"].astype("string").notna()
        & ~df["Encounter_ID"].astype("string").str.fullmatch(r"E\d{6}", na=False)
    ).sum()
)

# Load encounter master for referential-integrity checks
encounters_df = pd.read_csv(CLEANED_DIR / "05_encounters_cleaned.csv")

valid_encounter_ids = set(encounters_df["Encounter_ID"].dropna())
encounter_patient_map = (
    encounters_df
    .drop_duplicates("Encounter_ID")
    .set_index("Encounter_ID")["Patient_ID"]
)

print("\nEncounter_IDs not found in encounters master:",
      (~df["Encounter_ID"].isin(valid_encounter_ids)).sum())

# Patient_ID should agree with the Patient_ID linked to Encounter_ID
expected_patient_id = df["Encounter_ID"].map(encounter_patient_map)

patient_mismatch_mask = (
    expected_patient_id.notna()
    & df["Patient_ID"].notna()
    & (df["Patient_ID"] != expected_patient_id)
)

print("Patient_ID / Encounter_ID mismatches:", patient_mismatch_mask.sum())

# Claim_Date
claim_dates_temp = pd.to_datetime(df["Claim_Date"], errors="coerce")
encounter_dates_temp = (
    df["Encounter_ID"].map(
        encounters_df.drop_duplicates("Encounter_ID")
        .set_index("Encounter_ID")["Encounter_Date"]
    )
)
encounter_dates_temp = pd.to_datetime(encounter_dates_temp, errors="coerce")

print("\nInvalid Claim_Date values:", claim_dates_temp.isna().sum())
print(
    "Claim_Date before Encounter_Date:",
    (
        claim_dates_temp.notna()
        & encounter_dates_temp.notna()
        & (claim_dates_temp < encounter_dates_temp)
    ).sum()
)

# Payer
print("\nPayer missing values:", df["Payer"].isna().sum())
print("Payer unique values:")
print(df["Payer"].value_counts(dropna=False))

print(
    "Payer values with leading/trailing whitespace:",
    (
        df["Payer"].astype("string")
        != df["Payer"].astype("string").str.strip()
    ).sum()
)

# Claim_Status
print("\nClaim_Status missing values:", df["Claim_Status"].isna().sum())
print("Claim_Status unique values before cleaning:")
print(df["Claim_Status"].value_counts(dropna=False))

print(
    "Claim_Status values with leading/trailing whitespace:",
    (
        df["Claim_Status"].astype("string")
        != df["Claim_Status"].astype("string").str.strip()
    ).sum()
)

# Financial amounts
for column in ["Claimed_Amount", "Approved_Amount", "Rejected_Amount"]:
    print(
        f"{column}: missing={df[column].isna().sum()}, "
        f"negative={(df[column] < 0).sum()}, "
        f"zero={(df[column] == 0).sum()}"
    )

# Amount consistency
amount_formula_ok = np.isclose(
    df["Approved_Amount"] + df["Rejected_Amount"],
    df["Claimed_Amount"],
    atol=0.01,
    rtol=0
)

print(
    "Claimed_Amount formula mismatches:",
    (~amount_formula_ok).sum()
)

Duplicate Claim_ID: 0
Invalid Claim_ID format: 0
Invalid Patient_ID format: 0
Invalid Encounter_ID format: 0

Encounter_IDs not found in encounters master: 0
Patient_ID / Encounter_ID mismatches: 8999

Invalid Claim_Date values: 0
Claim_Date before Encounter_Date: 4484

Payer missing values: 0
Payer unique values:
Payer
Ayushman Bharat    1322
Star Health        1317
Self Pay           1310
Government         1279
Corporate          1271
HDFC Ergo          1263
ICICI Lombard      1238
Name: count, dtype: int64
Payer values with leading/trailing whitespace: 0

Claim_Status missing values: 1568
Claim_Status unique values before cleaning:
Claim_Status
NaN                   1568
Approved              1538
Rejected              1505
approved              1493
Pending               1466
Partially Approved    1430
Name: count, dtype: int64
Claim_Status values with leading/trailing whitespace: 1493
Claimed_Amount: missing=0, negative=0, zero=0
Approved_Amount: missing=0, negative=0, zero=0
Rej

## 4. Data Cleaning

### 4.1 Patient_ID

In [8]:
# Use Encounter_ID as the reference because each encounter has a linked Patient_ID.
# Replace mismatched Patient_ID values with the Patient_ID associated with the encounter.
expected_patient_id = df["Encounter_ID"].map(
    encounters_df.drop_duplicates("Encounter_ID")
    .set_index("Encounter_ID")["Patient_ID"]
)

patient_id_cleaned = df["Patient_ID"].copy()
patient_id_cleaned.loc[expected_patient_id.notna()] = expected_patient_id.loc[
    expected_patient_id.notna()
]

df["Patient_ID"] = patient_id_cleaned

### 4.2 Claim_Date

In [9]:
claim_date_cleaned = pd.to_datetime(df["Claim_Date"], errors="coerce")

# A claim date earlier than the related encounter date cannot be reliably reconstructed.
# Convert only these unsupported dates to missing values.
encounter_date_reference = pd.to_datetime(
    df["Encounter_ID"].map(
        encounters_df.drop_duplicates("Encounter_ID")
        .set_index("Encounter_ID")["Encounter_Date"]
    ),
    errors="coerce"
)

invalid_claim_date_mask = (
    claim_date_cleaned.notna()
    & encounter_date_reference.notna()
    & (claim_date_cleaned < encounter_date_reference)
)

claim_date_cleaned.loc[invalid_claim_date_mask] = pd.NaT

df["Claim_Date"] = claim_date_cleaned

### 4.3 Claim_Status

In [10]:
claim_status_cleaned = df["Claim_Status"].str.strip()

# Standardize the capitalization/spacing while preserving missing statuses.
claim_status_cleaned = claim_status_cleaned.replace(
    {"approved": "Approved", "Approved ": "Approved"}
)

claim_status_cleaned = claim_status_cleaned.str.title()

# Restore the canonical multi-word status after title casing.
claim_status_cleaned = claim_status_cleaned.replace(
    {"Partially Approved": "Partially Approved"}
)

df["Claim_Status"] = claim_status_cleaned

## 5. Final Validation

In [11]:
# Basic validation
print("Final shape:", df.shape)
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate Claim_ID:", df["Claim_ID"].duplicated().sum())

# ID format validation
print(
    "Invalid Claim_ID after cleaning:",
    (
        df["Claim_ID"].astype("string").notna()
        & ~df["Claim_ID"].astype("string").str.fullmatch(r"C\d{6}", na=False)
    ).sum()
)

print(
    "Invalid Patient_ID after cleaning:",
    (
        df["Patient_ID"].astype("string").notna()
        & ~df["Patient_ID"].astype("string").str.fullmatch(r"P\d{6}", na=False)
    ).sum()
)

print(
    "Invalid Encounter_ID after cleaning:",
    (
        df["Encounter_ID"].astype("string").notna()
        & ~df["Encounter_ID"].astype("string").str.fullmatch(r"E\d{6}", na=False)
    ).sum()
)

# Referential integrity
expected_patient_id = df["Encounter_ID"].map(
    encounters_df.drop_duplicates("Encounter_ID")
    .set_index("Encounter_ID")["Patient_ID"]
)

print(
    "Patient_ID / Encounter_ID mismatches after cleaning:",
    (
        expected_patient_id.notna()
        & (df["Patient_ID"] != expected_patient_id)
    ).sum()
)

print(
    "Encounter_IDs not found in encounters master after cleaning:",
    (~df["Encounter_ID"].isin(valid_encounter_ids)).sum()
)

# Date validation
encounter_date_reference = pd.to_datetime(
    df["Encounter_ID"].map(
        encounters_df.drop_duplicates("Encounter_ID")
        .set_index("Encounter_ID")["Encounter_Date"]
    ),
    errors="coerce"
)

print(
    "Claim_Date before Encounter_Date after cleaning:",
    (
        df["Claim_Date"].notna()
        & encounter_date_reference.notna()
        & (df["Claim_Date"] < encounter_date_reference)
    ).sum()
)

# Claim status validation
print("\nClaim_Status values after cleaning:")
print(df["Claim_Status"].value_counts(dropna=False))

# Financial validation
print("\nNegative Claimed_Amount:", (df["Claimed_Amount"] < 0).sum())
print("Negative Approved_Amount:", (df["Approved_Amount"] < 0).sum())
print("Negative Rejected_Amount:", (df["Rejected_Amount"] < 0).sum())

formula_check = np.isclose(
    df["Approved_Amount"] + df["Rejected_Amount"],
    df["Claimed_Amount"],
    atol=0.01,
    rtol=0
)

print(
    "Claimed_Amount formula mismatches after cleaning:",
    (~formula_check).sum()
)

print("\nFinal dtypes:")
print(df.dtypes)

print("\nFinal missing values:")
print(df.isna().sum())

Final shape: (9000, 9)
Duplicate rows: 0
Duplicate Claim_ID: 0
Invalid Claim_ID after cleaning: 0
Invalid Patient_ID after cleaning: 0
Invalid Encounter_ID after cleaning: 0
Patient_ID / Encounter_ID mismatches after cleaning: 0
Encounter_IDs not found in encounters master after cleaning: 0
Claim_Date before Encounter_Date after cleaning: 0

Claim_Status values after cleaning:
Claim_Status
Approved              3031
NaN                   1568
Rejected              1505
Pending               1466
Partially Approved    1430
Name: count, dtype: int64

Negative Claimed_Amount: 0
Negative Approved_Amount: 0
Negative Rejected_Amount: 0
Claimed_Amount formula mismatches after cleaning: 0

Final dtypes:
Claim_ID                   object
Patient_ID                 object
Encounter_ID               object
Claim_Date         datetime64[ns]
Payer                      object
Claimed_Amount            float64
Approved_Amount           float64
Rejected_Amount           float64
Claim_Status           

## 6. Export Cleaned Data

In [13]:
df.to_csv(CLEANED_DIR / "11_claims_cleaned.csv", index=False)